### set up Jenkins in K8s

#### Install jenkins by helm
```bash
 # create new namespace
 kubectl create namespace jenkins

 # download default values.yaml file from jenkins helm chart
 helm show values jenkins/jenkins > jenkins_values.yaml

  # you can edit and change settings in values.yaml and then update jenkins installation
 helm install jenkins jenkins/jenkins -n jenkins -f jenkins_values.yaml

 # or directly deploy jenkins with default values to jenkins namespace and then update
 helm install jenkins jenkins/jenkins -n jenkins 
 helm upgrade jenkins jenkins/jenkins -f jenkins_values.yaml -n jenkins

 # list all installed helm release
 helm list -- all-namespaces

 # show the status of an installed release with release name
 # use this to get the random jenkins password generated by installation
 helm status jenkins -n jenkins 
 ```

#### Set up ingress
* identify the ingress controller you want to use
* refer to jenkins_ingress.yaml file for ingress
* run `kubectl apply -f jenkins_ingress.yaml`
* access jenkins 

#### Set up credentials and host key to connect to github and bitbucket
* generate host keys for GitHub and bitbucket
    ```bash
    # Example of github and bitbucket
    ssh-keyscan github.com >> known_hosts
    ssh-keyscan bitbucket.org >> known_hosts
    ```
* add host key to Jenkins
  + Open Jenkins UI
  + Log in to your Jenkins instance
  + Go to Manage Jenkins (usually in the left sidebar)
    + click Security
    + find and click Git Host Key Verification Configuration
 
* in Configure Host Key Verification
  + Select "Manually provided keys" as the strategy
  + In the Approved Host Keys section, paste the line you copied from ssh-keyscan, click Save
    + You might see output in known_hosts like:
    ```text
    bitbucket.org:22 SSH-2.0-OpenSSH_8.2p1 Ubuntu-4ubuntu0.5
    bitbucket.org ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIE...
    ```
    Only copy this line: `bitbucket.org ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIE...`
* Generate SSH key of bitbucket/github for jenkins
  + `ssh-keygen -t ed25519` and when prompt the file to save key, use /home/<your_home_folder>/.ssh/id\_ed25519
  + the public and private key files are id\_ed25519.pub and id\_ed25519, respectively
* Add public key to bitbucket
  + copy public key file's content by `cat /home/<your_home_folder>/.ssh/id_ed25519.pub`
  + go to bitbucket -> settings -> add SSH key -> add workspace key and paste the public key
* Add private key from jenkins GUI
  + in jenkins GUI -> manage Jenkins -> Credentials -> add SSH Username with private key
    + input 'git' in Username
    + in Private Key -> Enter directly, copy and paste the entire content of id\_ed25519 include ---begin and --end lines
* test using a pipeline and build it
  ```groovy
  pipeline {
      agent any

      stages {
          stage('Checkout') {
              steps {
                  git credentialsId: 'bitbucket_ssh',
                  url: 'git@bitbucket.org:<org>/<repo name>.git',
                  branch: 'main'
              }
          }

          stage('Verify') {
              steps {
                  sh 'ls -la'
              }
          }
      }
  }
  ```
  

#### Set up pipeline from jenkins GUI using kubernetes agent
* define pipeline script from jenkins GUI
* jenkins in k8s doesn't have agent: {docker {image: 'python:3.10'}} option. Only has any, kubernetes
* jenkins in k8s don't config worker nodes, and therefore, no label nodes, which is used in typical jenkins config
  + in typical jenkins
    + make sure the worker node has installed a virtual env with all packages
    + define worker node in agent { label 'worker node name'}
    + you define worker node name and config in jenkins (see set up local jenkins link) 
```groovy
pipeline {
  agent {
    kubernetes {
      defaultContainer 'python'  // set python as default container.
      // must use command: -cat and tty:true to make container alive for jenkins to login to python container
        yaml """
    apiVersion: v1
    kind: pod
    spec:
      containers:
        - name: python
          image: python:3.10
          command:
            - cat 
          tty: true
    """ 
    }
    }

    environment {
        VENV_DIR = 'venv'
    }

    stages {
      stage('Run') {
        steps {
          sh 'python --version'
        }
      }

      stage('Git Checkout') {
        steps {
          git credentialsId: 'bitbucket_ssh',
              url: 'git@github.com:yuanMLOps/LLM_NLP.git',
              branch: 'main'
        }
      }

      stage('Verify') {
        steps {
          sh 'ls-la'
        }
      }

      stage('Setup python Environment') {
        steps { // use . $VENV_DIR/bin/activate as source $VENV_DIR/bin/activate
          sh '''
          python3 -m venv $VENV_DIR
          . $VENV_DIR/bin/activate

          pip install -r requirements.txt
          '''
        }
      }

      stage('Run Tests') {
        steps {
          sh '''
          . venv/bin/activate 
          which pytest
          export PYTHONPATH=$(pwd)
          pytest
          '''
        }
      }

    }
  }
 ```

* refer the following link for jenkins project configuration
  [jenkins project config](setup_k8s_agent.docx)        

#### Set up pipeline by jenkinsfile in code repo using kubernetes agent
* define jenkinsfile in the root folder of code repo
  + don't need to include checkout stage in jenkinsfile since it will automatically check out the code
  + other parts are the same as pipeline script defined in jenkins GUI
  + you can't run jenkinsfile with docker as agent in typical/local jenkins since you can't specify which docker container to run pipeline reliably
```groovy
pipeline {
  agent {
    kubernetes {
      defaultContainer 'python'  // set python as default container.
      // must use command: - cat and tty:true to make container alive for jenkins to login to python container
      // we can also use - sleep infinity for the same purpose
        yaml """
    apiVersion: v1
    kind: pod
    spec:
      containers:
        - name: python
          image: python:3.10
          command:
            - sleep       # 
            - infinity
         
    """ 
    }
    }

    environment {
        VENV_DIR = 'venv'
    }

    stages {
      stage('Run') {
        steps {
          sh 'python --version'
        }
      }

     stage('Verify') {
       steps {
          sh 'ls-la'
        }
      }

      stage('Setup python Environment') {
        steps { // use . $VENV_DIR/bin/activate as source $VENV_DIR/bin/activate
          sh '''
          python3 -m venv $VENV_DIR
          . $VENV_DIR/bin/activate

          pip install -r requirements.txt
          '''
        }
      }

      stage('Run Tests') {
        steps {
          sh '''
          . venv/bin/activate 
          which pytest
          export PYTHONPATH=$(pwd)
          pytest
          '''
        }
      }

    }
  }
 ```
* for configuration of project in jenkins, [refer to this link](setup_k8s_jenkinsfile.docx)